# §12.6.5 — 시퀀스 길이별 성능으로 확인하는 병목

> 딥러닝 교재 · 3부 12장 6절 5항 (🐍)
> 선행: §12.6.1(고정 문맥의 병목) · §12.6.2(질의·키·값의 원형) · §12.6.3(소프트 검색 독법)

## 이 노트북이 답하는 질문

1. **고정 길이 문맥의 성능은 입력 길이 $S$와 함께 정말 무너지는가?** 어텐션은 버티는가.
2. **무너짐의 원인은 학습 실패인가 용량인가?** 상태 크기 $m$을 키워 무너지는 지점의 이동을 본다.
3. **어텐션은 실제로 조회를 배우는가?** 학습된 정렬 행렬에서 질량이 정답 위치로 몰리는지 확인한다.

**예상 실행 시간** CPU 약 4분 (`FAST = True`이면 약 1분).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 합성 회상 과제 — 병목이 정확히 드러나도록

키–값 쌍 $S$개의 열을 입력하고, 마지막에 질의 키 하나를 주어 대응하는 값을 답하게
한다(§12.6.5). 키는 $N_K=24$개, 값은 $N_V=8$부류에서 뽑고, 한 시행 안에서 키는 겹치지
않는다. 우연 수준은 $1/N_V=0.125$다.

설계를 두 가지로 통제한다. 첫째, 각 시각의 입력은 [키 원-핫 $\,|\,$ **쌍으로 묶인**
원-핫 $e_k\otimes e_v$]다. 쌍을 미리 묶어 주는 이유는 실패 원인의 분리다 — "키와 값을
결합하는 학습"이 아니라 "고정 크기 요약"이 병목임을 보이려면, 인코더가 할 일을
**저장**만으로 줄여야 한다. 둘째, 질의 키는 시퀀스가 아니라 판독기에 직접 준다. 질의가
인코더 상태를 통과하며 열화되는 효과를 없애기 위해서다.

이 과제가 병목의 리트머스인 이유: 질의는 **모든 쌍을 읽은 뒤에야** 온다. 고정 문맥
모델은 어떤 키가 질의될지 모른 채 $S$쌍 전부를 상태 $c\in\mathbb{R}^m$에 욱여넣어야
한다. §12.6.1의 "무엇을 버릴지 옳게 정할 수 없는" 상황 그대로다.

In [ ]:
N_K, N_V = 24, 8
D_IN = N_K + N_K * N_V        # [키 원-핫 | 쌍(키⊗값) 원-핫]

def make_batch(S, B, rn):
    X = np.zeros((B, S, D_IN))
    keys = np.zeros((B, S), dtype=int)
    vals = rn.integers(0, N_V, size=(B, S))
    qpos = rn.integers(0, S, size=B)
    for b in range(B):
        keys[b] = rn.choice(N_K, size=S, replace=False)   # 키 중복 없음
    X[np.arange(B)[:, None], np.arange(S)[None, :], keys] = 1.0
    X[np.arange(B)[:, None], np.arange(S)[None, :], N_K + keys * N_V + vals] = 1.0
    qkey = keys[np.arange(B), qpos]
    Q = np.zeros((B, N_K)); Q[np.arange(B), qkey] = 1.0
    y = vals[np.arange(B), qpos]
    return X, Q, y, qpos

---
## 2. 두 모델 — 같은 인코더, 다른 판독

인코더는 두 모델이 공유하는 tanh 순환망이다. 다른 것은 판독뿐이다.

* **고정 문맥(seq2seq형)** — 마지막 상태 $c=h_S$ 하나로 답한다.
* **어텐션 부착** — 상태들 $h_1,...,h_S$를 보관해 두고, 질의 사영 $q=W_q e_{\text{질의}}$로
  곱셈 점수 $e_j=q^\top h_j$(§12.6.2)를 매겨 $c_{\text{att}}=\sum_j \alpha_j h_j$를 만든다.

판독은 둘 다 질의 조건부 양선형 $\mathrm{logits}_v=e_{\text{질의}}^\top A_v\, c$로
통일한다. 질의마다 다른 선형 판독을 골라 쓰는 구조로, §12.6.2의 곱셈 점수와 같은
급의 상호작용을 판독에도 허락한 것이다. 두 모델의 차이는 오직 $c$가 **무엇이냐**
— 길이와 무관한 요약이냐, 질의로 조회한 결과냐 — 뿐이다.

In [ ]:
RHO_ENC = 0.9      # 인코더 초기 스펙트럼 반경

def init_model(m, attn, seed):
    rn = np.random.default_rng(seed)
    Wh = rn.standard_normal((m, m)) / np.sqrt(m)
    Wh *= RHO_ENC / np.abs(np.linalg.eigvals(Wh)).max()
    Wx = rn.standard_normal((D_IN, m)) / np.sqrt(D_IN) * 2.0
    b = np.zeros(m)
    A = rn.standard_normal((N_V, N_K, m)) / np.sqrt(m)    # 질의 조건부 양선형 판독
    bo = np.zeros(N_V)
    ps = [Wh, Wx, b, A, bo]
    if attn:
        ps.append(rn.standard_normal((N_K, m)) / np.sqrt(N_K))   # W_q
    return ps

def forward_backward(ps, X, Q, y, attn):
    Wh, Wx, b, A, bo = ps[:5]
    Wq = ps[5] if attn else None
    B, S, _ = X.shape; m = Wh.shape[0]
    H = np.zeros((B, S + 1, m)); Z = np.zeros((B, S, m))
    for t in range(S):
        Z[:, t] = H[:, t] @ Wh + X[:, t] @ Wx + b
        H[:, t + 1] = np.tanh(Z[:, t])
    Henc = H[:, 1:S + 1]                  # 보관된 인코더 상태 h_1..h_S
    if attn:
        qv = Q @ Wq                       # 질의 사영
        e = np.einsum('bm,bsm->bs', qv, Henc)
        e -= e.max(axis=1, keepdims=True)
        alpha = np.exp(e); alpha /= alpha.sum(axis=1, keepdims=True)
        mem = np.einsum('bs,bsm->bm', alpha, Henc)     # 조회된 문맥
    else:
        alpha, mem = None, H[:, S]                     # 고정 문맥
    logits = np.einsum('bi,vij,bj->bv', Q, A, mem) + bo
    logits -= logits.max(axis=1, keepdims=True)
    P = np.exp(logits); P /= P.sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(P[np.arange(B), y] + 1e-12))
    acc = np.mean(P.argmax(axis=1) == y)
    # ── 역전파 ──
    dlog = P.copy(); dlog[np.arange(B), y] -= 1.0; dlog /= B
    gA = np.einsum('bv,bi,bj->vij', dlog, Q, mem); gbo = dlog.sum(0)
    dmem = np.einsum('bv,vij,bi->bj', dlog, A, Q)
    dH_all = np.zeros((B, S + 1, m))
    if attn:
        dc = dmem
        dalpha = np.einsum('bm,bsm->bs', dc, Henc)
        dH_all[:, 1:S + 1] += alpha[:, :, None] * dc[:, None, :]
        de = alpha * (dalpha - (alpha * dalpha).sum(axis=1, keepdims=True))
        dqv = np.einsum('bs,bsm->bm', de, Henc)
        gWq = Q.T @ dqv
        dH_all[:, 1:S + 1] += de[:, :, None] * qv[:, None, :]
    else:
        gWq = None
        dH_all[:, S] += dmem
    gWh = np.zeros_like(Wh); gWx = np.zeros_like(Wx); gb = np.zeros_like(b)
    delta = dH_all[:, S]
    for t in range(S - 1, -1, -1):
        dz = delta * (1 - np.tanh(Z[:, t]) ** 2)
        gWh += H[:, t].T @ dz; gWx += X[:, t].T @ dz; gb += dz.sum(0)
        delta = dz @ Wh.T + dH_all[:, t]
    grads = [gWh, gWx, gb, gA, gbo] + ([gWq] if attn else [])
    return loss, acc, grads, alpha

def train_model(S, m, attn, seed=0, steps=None):
    steps = steps or (700 if FAST else 1500)
    ps = init_model(m, attn, seed)
    ms = [np.zeros_like(p) for p in ps]; vs = [np.zeros_like(p) for p in ps]
    rb = np.random.default_rng(5000 + seed)
    for t in range(1, steps + 1):
        X, Q, y, _ = make_batch(S, 64, rb)
        loss, acc, grads, _ = forward_backward(ps, X, Q, y, attn)
        for p_, g_, m_, v_ in zip(ps, grads, ms, vs):
            m_[:] = 0.9 * m_ + 0.1 * g_
            v_[:] = 0.999 * v_ + 0.001 * g_ * g_
            p_ -= 3e-3 * (m_ / (1 - 0.9 ** t)) / (np.sqrt(v_ / (1 - 0.999 ** t)) + 1e-8)
    re = np.random.default_rng(SEED + 9)
    X, Q, y, qpos = make_batch(S, 1000, re)
    _, acc, _, alpha = forward_backward(ps, X, Q, y, attn)
    return acc, (X, Q, y, qpos, alpha)

---
## 3. 길이 $S$를 훑는다 — 그리고 $m$도

(b)용: $m=16$으로 두 모델을 길이별로 학습한다. (c)용: 고정 문맥 모델만 $m$을 바꿔
같은 훑기를 반복한다. 실패의 원인이 용량이라면, $m$을 키울 때 무너지는 지점이
오른쪽으로 밀려야 한다.

In [ ]:
S_LIST = [2, 6, 12, 20] if FAST else [2, 4, 6, 8, 12, 16, 20]
M_BASE = 16
acc_fix, acc_att = [], []
keep_att = None
for S in S_LIST:
    a_f, _ = train_model(S, M_BASE, attn=False, seed=1)
    a_a, pack = train_model(S, M_BASE, attn=True, seed=1)
    acc_fix.append(a_f); acc_att.append(a_a)
    if S == S_LIST[-1]:
        keep_att = pack                     # (d)용 정렬 행렬
    print(f"S={S:2d}:  고정 문맥 {a_f:.3f}   어텐션 {a_a:.3f}   ({time.time()-_t0:.0f}초)")

In [ ]:
M_LIST = [8, 32] if FAST else [8, 32, 64]
acc_fix_m = {M_BASE: acc_fix}
for m in M_LIST:
    accs = []
    for S in S_LIST:
        a, _ = train_model(S, m, attn=False, seed=1)
        accs.append(a)
    acc_fix_m[m] = accs
    print(f"m={m:2d} (고정 문맥):  " + "  ".join(f"{a:.2f}" for a in accs) +
          f"   ({time.time()-_t0:.0f}초)")

---
## 4. 교재 그림 — fig_12_6_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 과제와 두 판독의 구조 — 도식
ax = axes[0]; ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 7)
S_demo = 5
hx = []
for j in range(S_demo):
    x0 = 0.5 + 1.30 * j
    ax.add_patch(plt.Rectangle((x0, 5.6), 1.1, 1.05, fc='#E8F4FB', ec='k', lw=0.8))
    ax.text(x0 + 0.55, 6.38, f'$k_{j+1}$', ha='center', va='center', fontsize=9)
    ax.text(x0 + 0.55, 5.9, f'$v_{j+1}$', ha='center', va='center', fontsize=9, color=CB[5])
    ax.annotate('', xy=(x0 + 0.55, 4.78), xytext=(x0 + 0.55, 5.55),
                arrowprops=dict(arrowstyle='->', color='0.55', lw=0.7))
    ax.add_patch(plt.Circle((x0 + 0.55, 4.38), 0.34, fc='#E8F7F0', ec=CB[3], lw=0.9))
    ax.text(x0 + 0.55, 4.38, f'$h_{j+1}$', ha='center', va='center', fontsize=8)
    hx.append(x0 + 0.55)
    if j < S_demo - 1:
        ax.annotate('', xy=(x0 + 1.5, 4.38), xytext=(x0 + 0.93, 4.38),
                    arrowprops=dict(arrowstyle='->', color=CB[3], lw=0.8))
qx = 0.5 + 1.30 * S_demo + 0.55
ax.add_patch(plt.Rectangle((qx, 5.6), 1.55, 1.05, fc='#FDEBD9', ec=CB[4], lw=1.2))
ax.text(qx + 0.775, 6.13, lab('질의 $k_3$?', 'query $k_3$?'), ha='center', va='center',
        fontsize=9, color=CB[4])
# 왼쪽 상자: 고정 문맥 — 질의를 모른 채 요약
ax.add_patch(plt.Rectangle((0.5, 2.25), 3.8, 1.05, fc='#FBF3DB', ec='k', lw=0.8))
ax.text(2.4, 3.03, lab('고정 문맥:  $c = h_S$', 'fixed: $c=h_S$'), ha='center', fontsize=9)
ax.text(2.4, 2.5, lab('질의를 모른 채 $S$쌍을 $m$차원에 요약', 'summarize blindly'),
        ha='center', fontsize=8, color=CB[4])
ax.annotate('', xy=(4.05, 3.35), xytext=(hx[-1], 3.98),
            arrowprops=dict(arrowstyle='->', color=CB[1], lw=1.3,
                            connectionstyle='arc3,rad=0.22'))
# 오른쪽 상자: 어텐션 — 보관한 상태를 질의로 조회
ax.add_patch(plt.Rectangle((5.4, 2.25), 4.2, 1.05, fc='#EAF3FA', ec='k', lw=0.8))
ax.text(7.5, 3.03, lab('어텐션:  $c_{\\rm att}=\\sum_j \\alpha_j h_j$', 'attention'),
        ha='center', fontsize=9)
ax.text(7.5, 2.5, lab('보관한 상태를 질의로 그때그때 조회', 'store & look up'),
        ha='center', fontsize=8, color=CB[5])
for j in range(S_demo):
    ax.annotate('', xy=(6.0 + 0.55 * j, 3.35), xytext=(hx[j], 3.98),
                arrowprops=dict(arrowstyle='-', color='0.72', lw=0.6))
ax.annotate('', xy=(8.6, 3.35), xytext=(qx + 0.775, 5.55),
            arrowprops=dict(arrowstyle='->', color=CB[4], lw=1.2,
                            connectionstyle='arc3,rad=-0.12'))
ax.text(9.15, 4.4, '$\\alpha$', color=CB[4], fontsize=10)
# 판독 화살표
ax.annotate('', xy=(2.4, 1.35), xytext=(2.4, 2.2),
            arrowprops=dict(arrowstyle='->', color='k', lw=0.9))
ax.annotate('', xy=(7.5, 1.35), xytext=(7.5, 2.2),
            arrowprops=dict(arrowstyle='->', color='k', lw=0.9))
ax.text(2.4, 1.05, '$\\hat y$', ha='center', fontsize=10)
ax.text(7.5, 1.05, '$\\hat y$', ha='center', fontsize=10)
ax.text(5.0, 0.3, lab('판독은 둘 다 질의 조건부 양선형 — 차이는 $c$뿐',
                      'same query-conditioned readout; only $c$ differs'),
        ha='center', fontsize=8)
ax.set_title(lab('(a) 회상 과제와 두 판독', '(a) task structure'), fontsize=10)

# (b) 길이별 정확도
ax = axes[1]
ax.plot(S_LIST, acc_fix, 'o-', color=CB[4], ms=5, label=lab('고정 문맥', 'fixed'))
ax.plot(S_LIST, acc_att, 's-', color=CB[5], ms=5, label=lab('어텐션', 'attention'))
ax.axhline(1 / N_V, color='k', lw=0.7, ls=':')
ax.text(S_LIST[-1], 1 / N_V + 0.02, lab('우연 수준', 'chance'), fontsize=8, ha='right')
ax.set_ylim(0, 1.05)
ax.set_xlabel(lab('키–값 쌍의 수 $S$', 'sequence length $S$'))
ax.set_ylabel(lab('회상 정확도', 'recall accuracy'))
ax.set_title(lab(f'(b) 길이에 따른 성능 ($m={M_BASE}$)', '(b) accuracy vs length'), fontsize=10)
ax.legend(fontsize=9)

# (c) 고정 문맥 모델의 m 사다리
ax = axes[2]
m_all = sorted(acc_fix_m.keys())
for i, m in enumerate(m_all):
    ax.plot(S_LIST, acc_fix_m[m], 'o-', color=CB[1 + i], ms=4, label=f'$m={m}$')
ax.axhline(1 / N_V, color='k', lw=0.7, ls=':')
ax.set_ylim(0, 1.05)
ax.set_xlabel(lab('키–값 쌍의 수 $S$', 'sequence length $S$'))
ax.set_ylabel(lab('회상 정확도 (고정 문맥)', 'recall accuracy (fixed)'))
ax.set_title(lab('(c) $m$을 키우면 무너지는 지점이 밀린다', '(c) capacity ladder'), fontsize=10)
ax.legend(fontsize=8)

# (d) 학습된 정렬 행렬
ax = axes[3]
Xd, Qd, yd, qposd, alphad = keep_att
n_show = 24
order = np.argsort(qposd[:n_show])
A_mat = alphad[:n_show][order]
imv = ax.imshow(A_mat, cmap='viridis', aspect='auto', vmin=0, vmax=A_mat.max())
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.scatter(qposd[:n_show][order], np.arange(n_show), s=18, facecolors='none',
           edgecolors='w', linewidths=1.0, label=lab('정답 키 위치', 'true position'))
ax.grid(False)
ax.set_xlabel(lab('입력 위치 $j$', 'input position $j$'))
ax.set_ylabel(lab('시행 (정답 위치로 정렬)', 'trial (sorted)'))
ax.set_title(lab(f'(d) 정렬 행렬 $\\alpha$ ($S={S_LIST[-1]}$)', '(d) alignment'), fontsize=10)
ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

save_book_fig(fig, 'fig_12_6_5')
plt.show()

> ### 읽는 법
>
> (b) 고정 문맥 모델은 길이와 함께 가파르게 무너지고, 같은 인코더에 판독만 바꾼 어텐션
> 모델은 버틴다. 판독의 차이가 곧 병목의 유무다.
> (c) 확인 사살 — $m$을 키우면 무너지는 지점이 오른쪽으로 밀린다. 실패의 원인이
> 학습이 아니라 **용량**이라는 뜻이다. 단, 사다리의 걸음마다 파라미터와 계산이 함께
> 자란다. 길이에 맞춰 $m$을 계속 키우는 것은 처방이 아니라 문제의 재진술이다.
> (d) 어텐션 질량이 질의된 키의 위치(흰 동그라미)로 몰린다. 모델이 실제로 **조회**를
> 배웠다는 확인이자, "그럴듯한 그림"과 "설명"의 간극(§12.6.7)이라는 경고의 예고편이다.
>
> 설계 주석 하나. 초기 실험에서 쌍을 묶지 않은 입력([키 원-핫 $|$ 값 원-핫])을 주면
> 고정 문맥 모델은 $S=2$에서조차 "가장 최근 값을 답하는" 지름길에 갇혀 우연+최신
> 수준(약 0.56)을 벗어나지 못했다. 키–값 **결합** 자체가 tanh 순환망에게 어려운 학습
> 문제라는 뜻이다. 본 실험이 쌍을 미리 묶어 주는 이유가 이것이다 — 결합의 어려움을
> 걷어내야 남는 것이 순수한 **요약 용량**의 병목이고, 그것이 §12.6.1의 논증이다.

---
## 5. 자기 점검

1. (b)에서 고정 문맥 모델이 무너지기 시작하는 $S$는 대략 얼마인가? $m=16$차원 상태가 담을 수 있는 쌍 수와 견주어 보라(§12.6.1의 $O(m)$비트 논증).
2. (d)의 정렬 행렬에서 질량이 정답 위치 주변으로 옅게 번진 시행들이 있다. 인코더 상태 $h_j$가 위치 $j$까지의 **접두 열 전체**의 함수(§12.1.2)라는 사실로 이유를 설명하라.
3. 질의 키를 판독기가 아니라 **첫 시각의 입력**으로 주면 고정 문맥 모델의 곡선은 어떻게 변하겠는가? 바꿔 실행해 보고, "무엇을 버릴지 미리 알 수 있는" 상황(§12.6.1)과 연결하라.
4. 어텐션 모델의 $W_q$를 학습하지 않고 무작위로 고정하면 성능이 얼마나 떨어지는가? 조회가 "배워지는" 것임을 확인하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `S_LIST` | 3절 | 최대 20 | 더 긴 열에서 어텐션도 무너지는지 |
| `M_BASE` | 3절 | 16 | 병목의 폭 |
| `N_V` | 1절 | 8 | 값 부류 수 — 쌍당 요구 비트 |
| `RHO_ENC` | 2절 | 0.9 | 인코더 기억의 감쇠율(§12.3) |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")